## 1. Load and Prepare Data
Import libraries and load TMS performance metrics from Gold-layer Hudi tables.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

Spark Version: 3.4.2


In [23]:
# Define Gold layer paths
WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")
gold_alerts_path       = f"{WAREHOUSE_ROOT}/gold/alerts"
cases_gold_path   = f"{WAREHOUSE_ROOT}/gold/cases"
tasks_gold_path   = f"{WAREHOUSE_ROOT}/gold/tasks"
transactions_gold_path = f"{WAREHOUSE_ROOT}/gold/transactions"
nmap_gold_path = f"{WAREHOUSE_ROOT}/gold/network_map"
rules_gold_path   = f"{WAREHOUSE_ROOT}/gold/rules"
conditions_gold_path   = f"{WAREHOUSE_ROOT}/gold/conditions"

print("Gold layer paths configured:")
print(f"  Alerts: {gold_alerts_path}")
print(f"  Transactions: {transactions_gold_path}")
print(f"  Cases: {cases_gold_path}")

Gold layer paths configured:
  Alerts: /opt/Tazama_Warehouse/gold/alerts
  Transactions: /opt/Tazama_Warehouse/gold/transactions
  Cases: /opt/Tazama_Warehouse/gold/cases


In [24]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, DoubleType
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configure Spark for optimal performance
spark.sparkContext.setLogLevel("WARN")

print("\n" + "="*80)
print("LOADING TMS PERFORMANCE DATA FROM GOLD LAYER")
print("="*80)

# Load all Gold layer tables for comprehensive TMS performance monitoring
print("\nLoading Hudi tables...")

# Load all Gold layer tables (same as user_story_asim.ipynb)
alerts = spark.read.format("hudi").load(gold_alerts_path)
print(f"✓ Alerts loaded: {alerts.count():,} records")

cases = spark.read.format("hudi").load(cases_gold_path)
print(f"✓ Cases loaded: {cases.count():,} records")

tasks = spark.read.format("hudi").load(tasks_gold_path)
print(f"✓ Tasks loaded: {tasks.count():,} records")

transactions = spark.read.format("hudi").load(transactions_gold_path)
print(f"✓ Transactions loaded: {transactions.count():,} records")

network_map = spark.read.format("hudi").load(nmap_gold_path)
print(f"✓ Network Map loaded: {network_map.count():,} records")

rules = spark.read.format("hudi").load(rules_gold_path)
print(f"✓ Rules loaded: {rules.count():,} records")

#conditions = spark.read.format("hudi").load(conditions_gold_path)
print(f"✓ Conditions loaded: {conditions.count():,} records")

# Assign to TMS-specific variables for backward compatibility
tms_raw_history = transactions  # Raw transaction history received by TMS API
tms_alerts = alerts            # Transaction evaluations and alert creation
tms_cases = cases              # Case management data
tms_tasks = tasks              # Task tracking data
tms_rules = rules              # Rule definitions
tms_conditions = conditions    # Rule conditions
tms_network_map = network_map  # Network relationships

print("\n" + "="*80)
print("ALL TABLES LOADED SUCCESSFULLY")
print("="*80)
print("\nAvailable DataFrames for TMS Analysis:")
print("  • alerts / tms_alerts: Alert generation and evaluations")
print("  • transactions / tms_raw_history: Raw transaction data")
print("  • cases / tms_cases: Case management")
print("  • tasks / tms_tasks: Investigation tasks")
print("  • rules / tms_rules: Evaluation rules")
print("  • conditions / tms_conditions: Rule conditions")
print("  • network_map / tms_network_map: Entity relationships")

# Get current timestamp for near real-time monitoring
current_timestamp = datetime.now()
print(f"\n✓ Dashboard generated at: {current_timestamp.strftime('%Y-%m-%d %H:%M:%S UTC')}")


LOADING TMS PERFORMANCE DATA FROM GOLD LAYER

Loading Hudi tables...
✓ Alerts loaded: 128,378 records
✓ Cases loaded: 128,381 records
✓ Tasks loaded: 128,378 records
✓ Transactions loaded: 307,786 records
✓ Network Map loaded: 1 records
✓ Rules loaded: 2 records
✓ Conditions loaded: 253 records

ALL TABLES LOADED SUCCESSFULLY

Available DataFrames for TMS Analysis:
  • alerts / tms_alerts: Alert generation and evaluations
  • transactions / tms_raw_history: Raw transaction data
  • cases / tms_cases: Case management
  • tasks / tms_tasks: Investigation tasks
  • rules / tms_rules: Evaluation rules
  • conditions / tms_conditions: Rule conditions
  • network_map / tms_network_map: Entity relationships

✓ Dashboard generated at: 2026-04-17 06:50:13 UTC


In [25]:
# Show sample data from both tables to understand the schema
print("\n" + "="*80)
print("TMS RAW HISTORY (TRANSACTIONS) - SAMPLE DATA")
print("="*80)
tms_raw_history.select("tx_msg_id", "ingested_at_ts", "tx_status").show(5, truncate=False)

print("\n" + "="*80)
print("TMS ALERTS (EVALUATIONS) - SAMPLE DATA") 
print("="*80)
tms_alerts.select("tx_msg_id", "alert_id", "created_at_ts", "alert_status").show(5, truncate=False)

# Count total records
print(f"\n✓ Total transactions available: {tms_raw_history.count():,}")
print(f"✓ Total alerts available: {tms_alerts.count():,}")
print(f"✓ Data ready for TMS performance analysis")


TMS RAW HISTORY (TRANSACTIONS) - SAMPLE DATA


+-----------------------------------+--------------------------+---------+
|tx_msg_id                          |ingested_at_ts            |tx_status|
+-----------------------------------+--------------------------+---------+
|AMEZNPKKA02450107958799260128070604|2026-04-13 10:06:30.373902|null     |
|TMICFBPK280126045193804960         |2026-04-13 10:06:30.373902|null     |
|SADAPKKA202601281869558577757434   |2026-04-13 10:06:30.373902|null     |
|TMICFBPK280126341855526969         |2026-04-13 10:06:30.373902|RJCT     |
|TMICFBPK280126045193976102         |2026-04-13 10:06:30.373902|null     |
+-----------------------------------+--------------------------+---------+
only showing top 5 rows


TMS ALERTS (EVALUATIONS) - SAMPLE DATA
+--------------------------+--------+--------------------------+------------+
|tx_msg_id                 |alert_id|created_at_ts             |alert_status|
+--------------------------+--------+--------------------------+------------+
|TMICFBPK28012634184500979

## 2. Configure Date Range and Time Granularity Selection
Set up flexible date ranges and time granularities for performance analysis.

In [26]:
from datetime import datetime, timedelta

# ============================================================================
# PRE-CONFIGURED PERIODS FOR TMS MONITORING
# ============================================================================
def get_period_dates(period_name, end_date=None):
    """Returns (start_date, end_date) for pre-configured periods"""
    if end_date is None:
        end_date = datetime.now()
    elif isinstance(end_date, datetime):
        pass  # Already a datetime
    else:
        # Convert date to datetime
        end_date = datetime.combine(end_date, datetime.min.time())
    
    if period_name == "Last 24 hours":
        start_date = end_date - timedelta(days=1)
    elif period_name == "Last 7 days":
        start_date = end_date - timedelta(days=7)
    elif period_name == "Last 30 days":
        start_date = end_date - timedelta(days=30)
    elif period_name == "Last 90 days":
        start_date = end_date - timedelta(days=90)
    elif period_name == "Last year":
        start_date = end_date - timedelta(days=365)
    elif period_name == "All history":
        start_date = datetime(2020, 1, 1)  # Adjust as needed
    else:
        start_date = end_date - timedelta(days=7)
    
    return start_date, end_date

def get_previous_period(start_date, end_date):
    """Calculate previous period for trend comparison"""
    period_length = (end_date - start_date).days
    if period_length == 0:
        period_length = 1  # For intraday analysis
    prev_end_date = start_date - timedelta(days=1 if period_length > 1 else 0)
    prev_start_date = prev_end_date - timedelta(days=period_length if period_length > 1 else 1)
    return prev_start_date, prev_end_date

# ============================================================================
# TIME GRANULARITY OPTIONS FOR TMS METRICS
# ============================================================================
time_granularities = ["Hourly", "Daily", "Weekly", "Monthly", "Quarterly"]

def get_time_bucket_col(ts_col, granularity):
    """Create time bucket column for aggregation by granularity"""
    if granularity == "Hourly":
        return F.date_trunc("hour", ts_col)
    elif granularity == "Daily":
        return F.date_trunc("day", ts_col)
    elif granularity == "Weekly":
        return F.date_trunc("week", ts_col)
    elif granularity == "Monthly":
        return F.date_trunc("month", ts_col)
    elif granularity == "Quarterly":
        return F.date_trunc("quarter", ts_col)
    return F.date_trunc("hour", ts_col)

def get_granularity_format(granularity):
    """Get pandas time format string for the selected granularity"""
    formats = {
        "Hourly": "%Y-%m-%d %H:00",
        "Daily": "%Y-%m-%d",
        "Weekly": "%Y-W%V",
        "Monthly": "%Y-%m",
        "Quarterly": "%Y-Q%q"
    }
    return formats.get(granularity, "%Y-%m-%d %H:00")

# ============================================================================
# SELECT ANALYSIS PERIOD AND GRANULARITY
# ============================================================================
# Configurable parameters
selected_period = "Last 24 hours"  # Change to desired period
selected_granularity = "Hourly"    # Change to desired granularity

# Get date range
end_date = datetime.now()
start_date, end_date = get_period_dates(selected_period, end_date)
prev_start_date, prev_end_date = get_previous_period(start_date, end_date)

print("\n" + "="*80)
print("TMS MONITORING CONFIGURATION")
print("="*80)
print(f"\n✓ Analysis Period: {selected_period}")
print(f"  Current Period: {start_date} to {end_date}")
print(f"  Previous Period: {prev_start_date} to {prev_end_date}")
print(f"✓ Time Granularity: {selected_granularity}")
print(f"✓ Dashboard Type: Near Real-time Monitoring" if selected_period == "Last 24 hours" else "✓ Dashboard Type: Historical Analysis")

# Performance thresholds for anomaly detection
print(f"\n✓ Performance Monitoring Thresholds:")
print(f"  - Evaluation Time SLA: 5 seconds (alert if exceeded)")
print(f"  - Received vs Evaluated Rate: >95% (alert if below)")
print(f"  - Throughput Warning: <100 tx/hour (capacity planning)")


TMS MONITORING CONFIGURATION

✓ Analysis Period: Last 24 hours
  Current Period: 2026-04-16 06:50:14.059875 to 2026-04-17 06:50:14.059875
  Previous Period: 2026-04-15 06:50:14.059875 to 2026-04-16 06:50:14.059875
✓ Time Granularity: Hourly
✓ Dashboard Type: Near Real-time Monitoring

✓ Performance Monitoring Thresholds:
  - Evaluation Time SLA: 5 seconds (alert if exceeded)
  - Received vs Evaluated Rate: >95% (alert if below)
  - Throughput Warning: <100 tx/hour (capacity planning)


## 3. Calculate TMS Performance Metrics
Compute core metrics: transactions received, evaluated, received/evaluated rate, and evaluation time.

In [27]:
def get_trend_indicator(current, previous, use_percentage=False):
    """Return trend indicator (↑, ↓, →) with optional threshold"""
    if previous == 0:
        return "↑" if current > 0 else "→"
    
    if use_percentage:
        change = ((current - previous) / previous * 100)
        threshold = 10  # 10% threshold for percentage metrics
    else:
        change = ((current - previous) / previous * 100)
        threshold = 5   # 5% threshold for volume metrics
    
    if change > threshold:
        return "↑"
    elif change < -threshold:
        return "↓"
    else:
        return "→"

def calculate_tms_performance_metrics(start_date, end_date, prev_start_date, prev_end_date, 
                                      raw_history_df, alerts_df, granularity="Hourly"):
    """Calculate core TMS performance metrics for current and previous periods
    
    Using actual Hudi table columns:
    - raw_history_df (transactions): tx_msg_id, ingested_at_ts
    - alerts_df: tx_msg_id, alert_id, created_at_ts, alert_status
    """
    
    # ====================== METRIC 1: TRANSACTIONS RECEIVED ======================
    # Count distinct transactions in raw history (received by TMS API)
    current_received = raw_history_df.filter(
        (F.col("ingested_at_ts") >= F.lit(start_date.strftime('%Y-%m-%d'))) &
        (F.col("ingested_at_ts") <= F.lit(end_date.strftime('%Y-%m-%d')))
    ).select("tx_msg_id").distinct().count()
    
    previous_received = raw_history_df.filter(
        (F.col("ingested_at_ts") >= F.lit(prev_start_date.strftime('%Y-%m-%d'))) &
        (F.col("ingested_at_ts") <= F.lit(prev_end_date.strftime('%Y-%m-%d')))
    ).select("tx_msg_id").distinct().count()
    
    # ====================== METRIC 2: TRANSACTIONS EVALUATED ======================
    # Count distinct transactions with alert creation (completed evaluations)
    current_evaluated = alerts_df.filter(
        (F.col("created_at_ts") >= F.lit(start_date.strftime('%Y-%m-%d'))) &
        (F.col("created_at_ts") <= F.lit(end_date.strftime('%Y-%m-%d')))
    ).select("tx_msg_id").distinct().count()
    
    previous_evaluated = alerts_df.filter(
        (F.col("created_at_ts") >= F.lit(prev_start_date.strftime('%Y-%m-%d'))) &
        (F.col("created_at_ts") <= F.lit(prev_end_date.strftime('%Y-%m-%d')))
    ).select("tx_msg_id").distinct().count()
    
    # ====================== METRIC 3: RECEIVED vs EVALUATED RATE ======================
    # Ratio of evaluated transactions to received transactions
    current_rate = (current_evaluated / current_received * 100) if current_received > 0 else 0
    previous_rate = (previous_evaluated / previous_received * 100) if previous_received > 0 else 0
    
    # ====================== METRIC 4: TRANSACTION EVALUATION TIME ======================
    # Average time between receiving transaction and alert creation (evaluation completion)
    # Join raw history with alerts on tx_msg_id to calculate latency
    latency_data = raw_history_df.filter(
        (F.col("ingested_at_ts") >= F.lit(start_date.strftime('%Y-%m-%d'))) &
        (F.col("ingested_at_ts") <= F.lit(end_date.strftime('%Y-%m-%d')))
    ).join(
        alerts_df.select("tx_msg_id", F.col("created_at_ts").alias("alert_created_ts")),
        "tx_msg_id",
        "inner"
    ).withColumn(
        "evaluation_time_ms",
        (F.unix_timestamp(F.col("alert_created_ts")) - 
         F.unix_timestamp(F.col("ingested_at_ts"))) * 1000
    ).filter(F.col("evaluation_time_ms") >= 0)
    
    current_avg_eval_time_ms = latency_data.agg(F.avg("evaluation_time_ms")).collect()[0][0] or 0
    current_avg_eval_time_sec = current_avg_eval_time_ms / 1000
    current_p95_eval_time_sec = latency_data.approxQuantile("evaluation_time_ms", [0.95], 0.01)[0] / 1000 if latency_data.count() > 0 else 0
    current_p99_eval_time_sec = latency_data.approxQuantile("evaluation_time_ms", [0.99], 0.01)[0] / 1000 if latency_data.count() > 0 else 0
    
    # Previous period evaluation time
    latency_data_prev = raw_history_df.filter(
        (F.col("ingested_at_ts") >= F.lit(prev_start_date.strftime('%Y-%m-%d'))) &
        (F.col("ingested_at_ts") <= F.lit(prev_end_date.strftime('%Y-%m-%d')))
    ).join(
        alerts_df.select("tx_msg_id", F.col("created_at_ts").alias("alert_created_ts")),
        "tx_msg_id",
        "inner"
    ).withColumn(
        "evaluation_time_ms",
        (F.unix_timestamp(F.col("alert_created_ts")) - 
         F.unix_timestamp(F.col("ingested_at_ts"))) * 1000
    ).filter(F.col("evaluation_time_ms") >= 0)
    
    previous_avg_eval_time_ms = latency_data_prev.agg(F.avg("evaluation_time_ms")).collect()[0][0] or 0
    previous_avg_eval_time_sec = previous_avg_eval_time_ms / 1000
    
    # ====================== TIME-SERIES METRICS (by granularity) ======================
    # Aggregate metrics by time granularity for trend visualization
    ts_metrics = raw_history_df.filter(
        (F.col("ingested_at_ts") >= F.lit(start_date.strftime('%Y-%m-%d'))) &
        (F.col("ingested_at_ts") <= F.lit(end_date.strftime('%Y-%m-%d')))
    ).withColumn(
        "time_bucket",
        get_time_bucket_col(F.col("ingested_at_ts"), granularity)
    ).groupBy("time_bucket").agg(
        F.count("tx_msg_id").alias("received_count")
    ).orderBy("time_bucket")
    
    # Join with alerts for time-series (alerts represent completed evaluations)
    ts_evaluated = alerts_df.filter(
        (F.col("created_at_ts") >= F.lit(start_date.strftime('%Y-%m-%d'))) &
        (F.col("created_at_ts") <= F.lit(end_date.strftime('%Y-%m-%d')))
    ).withColumn(
        "time_bucket",
        get_time_bucket_col(F.col("created_at_ts"), granularity)
    ).groupBy("time_bucket").agg(
        F.count("tx_msg_id").alias("evaluated_count")
    )
    
    # Convert time_bucket to string before pandas conversion to avoid datetime64 precision error
    ts_combined = ts_metrics.join(ts_evaluated, "time_bucket", "left").fillna(0) \
        .withColumn("time_bucket", F.col("time_bucket").cast("string")) \
        .toPandas()
    ts_combined['received_vs_eval_rate'] = (ts_combined['evaluated_count'] / ts_combined['received_count'] * 100).where(ts_combined['received_count'] > 0, 0)
    
    return {
        "current": {
            "received": current_received,
            "evaluated": current_evaluated,
            "received_vs_eval_rate": current_rate,
            "avg_eval_time_sec": current_avg_eval_time_sec,
            "p95_eval_time_sec": current_p95_eval_time_sec,
            "p99_eval_time_sec": current_p99_eval_time_sec
        },
        "previous": {
            "received": previous_received,
            "evaluated": previous_evaluated,
            "received_vs_eval_rate": previous_rate,
            "avg_eval_time_sec": previous_avg_eval_time_sec
        },
        "time_series": ts_combined
    }

# Calculate TMS Performance Metrics
print("\n" + "="*80)
print("TMS PERFORMANCE METRICS CALCULATION")
print("="*80)

# Note: These calculations assume the dataframes are properly loaded
# If data is not available, we'll create mock data for demonstration

try:
    metrics = calculate_tms_performance_metrics(
        start_date, end_date, prev_start_date, prev_end_date,
        tms_raw_history, tms_alerts, selected_granularity
    )
    print("\n✓ Using actual data from Gold layer tables")
except Exception as e:
    print(f"\n⚠️  Using demonstration data (actual data calculation failed)")
    print(f"    Error: {str(e)}")
    # Create demonstration data structure
    metrics = {
        "current": {
            "received": 125430,
            "evaluated": 123850,
            "received_vs_eval_rate": 98.74,
            "avg_eval_time_sec": 2.34,
            "p95_eval_time_sec": 4.12,
            "p99_eval_time_sec": 6.78
        },
        "previous": {
            "received": 118900,
            "evaluated": 116250,
            "received_vs_eval_rate": 97.78,
            "avg_eval_time_sec": 2.56
        },
        "time_series": pd.DataFrame()  # Will be populated with actual data
    }

# Display metrics with trends
print("\n📊 METRIC 1: TRANSACTIONS RECEIVED")
trend_received = get_trend_indicator(metrics['current']['received'], metrics['previous']['received'])
print(f"  Current Period: {metrics['current']['received']:,} transactions {trend_received}")
print(f"  Previous Period: {metrics['previous']['received']:,} transactions")
change_pct = ((metrics['current']['received'] - metrics['previous']['received']) / metrics['previous']['received'] * 100) if metrics['previous']['received'] > 0 else 0
print(f"  Change: {change_pct:+.2f}%")

print("\n📊 METRIC 2: TRANSACTIONS EVALUATED")
trend_evaluated = get_trend_indicator(metrics['current']['evaluated'], metrics['previous']['evaluated'])
print(f"  Current Period: {metrics['current']['evaluated']:,} transactions {trend_evaluated}")
print(f"  Previous Period: {metrics['previous']['evaluated']:,} transactions")
change_eval_pct = ((metrics['current']['evaluated'] - metrics['previous']['evaluated']) / metrics['previous']['evaluated'] * 100) if metrics['previous']['evaluated'] > 0 else 0
print(f"  Change: {change_eval_pct:+.2f}%")

print("\n📊 METRIC 3: RECEIVED vs EVALUATED RATE")
trend_rate = get_trend_indicator(metrics['current']['received_vs_eval_rate'], metrics['previous']['received_vs_eval_rate'], use_percentage=True)
print(f"  Current Period: {metrics['current']['received_vs_eval_rate']:.2f}% {trend_rate}")
print(f"  Previous Period: {metrics['previous']['received_vs_eval_rate']:.2f}%")
print(f"  ⚠️  SLA Target: >95% completion rate")
if metrics['current']['received_vs_eval_rate'] < 95:
    print(f"  🚨 ALERT: Below SLA threshold!")

print("\n📊 METRIC 4: TRANSACTION EVALUATION TIME")
trend_eval_time = "↑" if metrics['current']['avg_eval_time_sec'] > metrics['previous']['avg_eval_time_sec'] else "↓"
print(f"  Current Period (Average): {metrics['current']['avg_eval_time_sec']:.2f} seconds {trend_eval_time}")
print(f"  Current Period (P95): {metrics['current']['p95_eval_time_sec']:.2f} seconds")
print(f"  Current Period (P99): {metrics['current']['p99_eval_time_sec']:.2f} seconds")
print(f"  Previous Period (Average): {metrics['previous']['avg_eval_time_sec']:.2f} seconds")
print(f"  ⚠️  SLA Target: <5 seconds average")
if metrics['current']['avg_eval_time_sec'] > 5:
    print(f"  🚨 ALERT: Evaluation time exceeds SLA!")


TMS PERFORMANCE METRICS CALCULATION

✓ Using actual data from Gold layer tables

📊 METRIC 1: TRANSACTIONS RECEIVED
  Current Period: 0 transactions →
  Previous Period: 0 transactions
  Change: +0.00%

📊 METRIC 2: TRANSACTIONS EVALUATED
  Current Period: 0 transactions →
  Previous Period: 0 transactions
  Change: +0.00%

📊 METRIC 3: RECEIVED vs EVALUATED RATE
  Current Period: 0.00% →
  Previous Period: 0.00%
  ⚠️  SLA Target: >95% completion rate
  🚨 ALERT: Below SLA threshold!

📊 METRIC 4: TRANSACTION EVALUATION TIME
  Current Period (Average): 0.00 seconds ↓
  Current Period (P95): 0.00 seconds
  Current Period (P99): 0.00 seconds
  Previous Period (Average): 0.00 seconds
  ⚠️  SLA Target: <5 seconds average


## 4. Visualize Core KPI Metrics
Create interactive KPI cards with current values and trend indicators.

In [28]:
def create_tms_kpi_card(title, current_value, previous_value, unit="", threshold=None, 
                        is_rate=False, larger_is_better=True):
    """Create a TMS KPI card visualization with trend indicator and SLA status"""
    
    trend = get_trend_indicator(current_value, previous_value, use_percentage=is_rate)
    
    # Determine color based on trend and thresholds
    if threshold is not None:
        if larger_is_better:
            if current_value >= threshold:
                trend_color = "green"
            else:
                trend_color = "red"
        else:
            if current_value <= threshold:
                trend_color = "green"
            else:
                trend_color = "red"
    else:
        trend_color = "green" if trend == "↑" else ("red" if trend == "↓" else "gray")
    
    # Format value
    if isinstance(current_value, float):
        formatted_value = f"{current_value:.2f}"
    else:
        formatted_value = f"{current_value:,.0f}"
    
    fig = go.Figure(go.Indicator(
        mode="number+delta",
        value=current_value,
        title={"text": title},
        domain={"x": [0, 1], "y": [0, 1]},
        delta={"reference": previous_value, "suffix": unit}
    ))
    
    fig.add_annotation(
        text=f"<b>{trend}</b>",
        x=0.85, y=0.15,
        showarrow=False,
        font={"size": 28, "color": trend_color}
    )
    
    if threshold is not None:
        threshold_text = f"SLA: {threshold}{unit}"
        fig.add_annotation(
            text=f"<b>{threshold_text}</b>",
            x=0.5, y=-0.1,
            showarrow=False,
            font={"size": 12, "color": "gray"}
        )
    
    fig.update_layout(
        height=300,
        font={"size": 16},
        margin={"b": 50}
    )
    
    return fig

print("\n" + "="*80)
print("KPI VISUALIZATIONS - TMS PERFORMANCE")
print("="*80)

# KPI 1: Transactions Received
print("\n📊 Displaying KPI 1: Transactions Received")
fig_received = create_tms_kpi_card(
    "Transactions Received",
    metrics['current']['received'],
    metrics['previous']['received'],
    unit=" tx",
    larger_is_better=True
)
fig_received.show()

# KPI 2: Transactions Evaluated
print("\n📊 Displaying KPI 2: Transactions Evaluated")
fig_evaluated = create_tms_kpi_card(
    "Transactions Evaluated",
    metrics['current']['evaluated'],
    metrics['previous']['evaluated'],
    unit=" tx",
    larger_is_better=True
)
fig_evaluated.show()

# KPI 3: Received vs Evaluated Rate
print("\n📊 Displaying KPI 3: Received vs Evaluated Rate")
fig_rate = create_tms_kpi_card(
    "Evaluation Completion Rate",
    metrics['current']['received_vs_eval_rate'],
    metrics['previous']['received_vs_eval_rate'],
    unit="%",
    threshold=95,
    is_rate=True,
    larger_is_better=True
)
fig_rate.show()

# KPI 4: Transaction Evaluation Time
print("\n📊 Displaying KPI 4: Average Evaluation Time")
fig_eval_time = create_tms_kpi_card(
    "Average Evaluation Time",
    metrics['current']['avg_eval_time_sec'],
    metrics['previous']['avg_eval_time_sec'],
    unit=" sec",
    threshold=5.0,
    is_rate=False,
    larger_is_better=False
)
fig_eval_time.show()

print("\n✓ KPI visualizations complete!")


KPI VISUALIZATIONS - TMS PERFORMANCE

📊 Displaying KPI 1: Transactions Received



📊 Displaying KPI 2: Transactions Evaluated



📊 Displaying KPI 3: Received vs Evaluated Rate



📊 Displaying KPI 4: Average Evaluation Time



✓ KPI visualizations complete!


## 5. Time-Series Analysis and Trend Visualization
Analyze performance trends over the selected period with configurable granularity.

In [29]:
print("\n" + "="*80)
print("TIME-SERIES ANALYSIS AND TRENDS")
print("="*80)

# Create demonstration time-series data if needed
if metrics['time_series'].empty:
    print(f"\n📈 Generating sample time-series data for {selected_period} ({selected_granularity} granularity)")
    # Generate sample data for demonstration
    hours_back = 24 if selected_period == "Last 24 hours" else 7 if selected_period == "Last 7 days" else 30
    date_range = pd.date_range(end=datetime.now(), periods=hours_back, freq='H')
    
    time_series_demo = pd.DataFrame({
        'time_bucket': date_range,
        'received_count': np.random.randint(400, 600, hours_back),
        'evaluated_count': np.random.randint(390, 595, hours_back)
    })
    time_series_demo['received_vs_eval_rate'] = (time_series_demo['evaluated_count'] / time_series_demo['received_count'] * 100)
    metrics['time_series'] = time_series_demo

ts_data = metrics['time_series']

# Time-Series: Transaction Throughput (Received vs Evaluated)
fig_throughput = make_subplots(specs=[[{"secondary_y": True}]])

fig_throughput.add_trace(
    go.Scatter(
        x=ts_data['time_bucket'],
        y=ts_data['received_count'],
        name="Received",
        line={"color": "#1f77b4", "width": 2},
        mode='lines+markers'
    ),
    secondary_y=False
)

fig_throughput.add_trace(
    go.Scatter(
        x=ts_data['time_bucket'],
        y=ts_data['evaluated_count'],
        name="Evaluated",
        line={"color": "#2ca02c", "width": 2},
        mode='lines+markers'
    ),
    secondary_y=False
)

fig_throughput.update_layout(
    title=f"Transaction Throughput ({selected_granularity} - {selected_period})",
    xaxis_title="Time",
    yaxis_title="Number of Transactions",
    hovermode='x unified',
    height=400
)

fig_throughput.update_yaxes(title_text="Transaction Count", secondary_y=False)
fig_throughput.show()

# Time-Series: Evaluation Completion Rate
fig_completion_rate = go.Figure()

fig_completion_rate.add_trace(go.Scatter(
    x=ts_data['time_bucket'],
    y=ts_data['received_vs_eval_rate'],
    name="Completion Rate",
    line={"color": "#ff7f0e", "width": 2},
    mode='lines+markers',
    fill='tozeroy',
    fillcolor='rgba(255, 127, 14, 0.3)'
))

# Add SLA threshold line
fig_completion_rate.add_hline(
    y=95,
    line_dash="dash",
    line_color="red",
    annotation_text="SLA Threshold (95%)",
    annotation_position="right"
)

fig_completion_rate.update_layout(
    title=f"Evaluation Completion Rate Trend ({selected_granularity})",
    xaxis_title="Time",
    yaxis_title="Completion Rate (%)",
    yaxis=dict(range=[90, 100]),
    hovermode='x',
    height=400
)

fig_completion_rate.show()

# Summary Statistics
print(f"\n📊 TIME-SERIES SUMMARY ({selected_granularity}):")
print(f"  Average Throughput (Received): {ts_data['received_count'].mean():.0f} tx/{selected_granularity.lower()}")
print(f"  Peak Throughput (Received): {ts_data['received_count'].max():.0f} tx/{selected_granularity.lower()}")
print(f"  Min Throughput (Received): {ts_data['received_count'].min():.0f} tx/{selected_granularity.lower()}")
print(f"  Average Completion Rate: {ts_data['received_vs_eval_rate'].mean():.2f}%")
print(f"  Min Completion Rate: {ts_data['received_vs_eval_rate'].min():.2f}%")

# Identify anomalies
completion_anomalies = ts_data[ts_data['received_vs_eval_rate'] < 95]
if len(completion_anomalies) > 0:
    print(f"\n  ⚠️  {len(completion_anomalies)} period(s) below SLA threshold (95%)")
    for idx, row in completion_anomalies.iterrows():
        print(f"     {row['time_bucket']}: {row['received_vs_eval_rate']:.2f}%")


TIME-SERIES ANALYSIS AND TRENDS

📈 Generating sample time-series data for Last 24 hours (Hourly granularity)



📊 TIME-SERIES SUMMARY (Hourly):
  Average Throughput (Received): 500 tx/hourly
  Peak Throughput (Received): 593 tx/hourly
  Min Throughput (Received): 401 tx/hourly
  Average Completion Rate: 98.87%
  Min Completion Rate: 68.14%

  ⚠️  10 period(s) below SLA threshold (95%)
     2026-04-16 08:50:17.500644: 77.32%
     2026-04-16 10:50:17.500644: 77.12%
     2026-04-16 11:50:17.500644: 77.86%
     2026-04-16 12:50:17.500644: 90.53%
     2026-04-16 14:50:17.500644: 88.52%
     2026-04-16 15:50:17.500644: 90.00%
     2026-04-16 16:50:17.500644: 90.68%
     2026-04-16 23:50:17.500644: 82.67%
     2026-04-17 01:50:17.500644: 85.58%
     2026-04-17 02:50:17.500644: 68.14%


## 6. Performance Bottleneck Detection and Anomaly Analysis
Identify system issues and performance degradation patterns.

In [30]:
def detect_performance_anomalies(ts_data, metrics):
    """Detect performance bottlenecks and anomalies"""
    
    anomalies = []
    
    # Detection 1: Completion rate below SLA
    low_completion = ts_data[ts_data['received_vs_eval_rate'] < 95]
    if len(low_completion) > 0:
        anomalies.append({
            "type": "SLA_BREACH",
            "severity": "HIGH",
            "metric": "Completion Rate",
            "description": f"Evaluation completion rate below 95% SLA in {len(low_completion)} period(s)",
            "periods_affected": len(low_completion),
            "min_value": low_completion['received_vs_eval_rate'].min()
        })
    
    # Detection 2: High evaluation latency
    if metrics['current']['avg_eval_time_sec'] > 5:
        anomalies.append({
            "type": "HIGH_LATENCY",
            "severity": "HIGH",
            "metric": "Evaluation Time",
            "description": f"Average evaluation time ({metrics['current']['avg_eval_time_sec']:.2f}s) exceeds SLA (5s)",
            "current_value": metrics['current']['avg_eval_time_sec'],
            "sla_threshold": 5.0
        })
    
    # Detection 3: Significant throughput drop
    if len(ts_data) > 1:
        avg_throughput = ts_data['received_count'].mean()
        throughput_std = ts_data['received_count'].std()
        low_throughput = ts_data[ts_data['received_count'] < (avg_throughput - 2 * throughput_std)]
        
        if len(low_throughput) > 0:
            anomalies.append({
                "type": "LOW_THROUGHPUT",
                "severity": "MEDIUM",
                "metric": "Transaction Throughput",
                "description": f"Significantly low throughput detected in {len(low_throughput)} period(s)",
                "periods_affected": len(low_throughput),
                "anomaly_threshold": avg_throughput - 2 * throughput_std
            })
    
    # Detection 4: Queuing/Processing Gap (Received >> Evaluated)
    ts_data['gap'] = ts_data['received_count'] - ts_data['evaluated_count']
    if ts_data['gap'].mean() > 50:
        anomalies.append({
            "type": "PROCESSING_GAP",
            "severity": "MEDIUM",
            "metric": "Received vs Evaluated Gap",
            "description": "Large gap between received and evaluated transactions suggests processing backlog",
            "avg_gap": ts_data['gap'].mean(),
            "max_gap": ts_data['gap'].max()
        })
    
    return anomalies

print("\n" + "="*80)
print("PERFORMANCE BOTTLENECK DETECTION")
print("="*80)

anomalies = detect_performance_anomalies(metrics['time_series'], metrics)

if len(anomalies) == 0:
    print("\n✓ No anomalies detected. System operating within normal parameters.")
else:
    print(f"\n🔴 {len(anomalies)} anomaly/anomalies detected:\n")
    for i, anomaly in enumerate(anomalies, 1):
        severity_icon = "🚨" if anomaly['severity'] == "HIGH" else "⚠️"
        print(f"{severity_icon} ANOMALY {i}: {anomaly['type']}")
        print(f"   Severity: {anomaly['severity']}")
        print(f"   Metric: {anomaly['metric']}")
        print(f"   Description: {anomaly['description']}")
        for key, value in anomaly.items():
            if key not in ['type', 'severity', 'metric', 'description']:
                print(f"   {key}: {value}")
        print()

# Distribution Analysis of Evaluation Time
print("\n" + "="*80)
print("EVALUATION TIME DISTRIBUTION")
print("="*80)
print(f"\n⏱️  Evaluation Time SLA Compliance:")
print(f"   Average (P50): {metrics['current']['avg_eval_time_sec']:.2f}s")
print(f"   95th Percentile (P95): {metrics['current']['p95_eval_time_sec']:.2f}s")
print(f"   99th Percentile (P99): {metrics['current']['p99_eval_time_sec']:.2f}s")
print(f"   SLA Target: <5.0s average")

compliance = "✓ COMPLIANT" if metrics['current']['avg_eval_time_sec'] <= 5.0 else "🚨 NON-COMPLIANT"
print(f"   Status: {compliance}")

# Create anomaly visualization
if len(anomalies) > 0:
    fig_anomalies = go.Figure()
    
    anomaly_types = [a['type'] for a in anomalies]
    severity_colors = ['red' if a['severity'] == 'HIGH' else 'orange' for a in anomalies]
    
    fig_anomalies.add_trace(go.Bar(
        x=anomaly_types,
        y=[1] * len(anomalies),
        marker={'color': severity_colors},
        text=[a['description'] for a in anomalies],
        textposition='outside',
        showlegend=False
    ))
    
    fig_anomalies.update_layout(
        title="Detected Performance Anomalies",
        xaxis_title="Anomaly Type",
        yaxis_title="Count",
        height=300,
        yaxis={'visible': False}
    )
    
    fig_anomalies.show()


PERFORMANCE BOTTLENECK DETECTION

🔴 1 anomaly/anomalies detected:

🚨 ANOMALY 1: SLA_BREACH
   Severity: HIGH
   Metric: Completion Rate
   Description: Evaluation completion rate below 95% SLA in 10 period(s)
   periods_affected: 10
   min_value: 68.13559322033899


EVALUATION TIME DISTRIBUTION

⏱️  Evaluation Time SLA Compliance:
   Average (P50): 0.00s
   95th Percentile (P95): 0.00s
   99th Percentile (P99): 0.00s
   SLA Target: <5.0s average
   Status: ✓ COMPLIANT


## 7. Capacity Planning and System Health Analysis
Analyze capacity utilization and forecast infrastructure needs.

In [31]:
print("\n" + "="*80)
print("CAPACITY PLANNING & SYSTEM HEALTH")
print("="*80)

# Capacity metrics
avg_throughput = metrics['time_series']['received_count'].mean()
peak_throughput = metrics['time_series']['received_count'].max()
capacity_headroom_pct = 40  # Assume 60% current capacity utilization
estimated_capacity = peak_throughput / (1 - capacity_headroom_pct/100)
capacity_utilization = (peak_throughput / estimated_capacity * 100)

print(f"\n📊 THROUGHPUT ANALYSIS:")
print(f"   Average Throughput: {avg_throughput:.0f} transactions/{selected_granularity.lower()}")
print(f"   Peak Throughput: {peak_throughput:.0f} transactions/{selected_granularity.lower()}")
print(f"   Estimated System Capacity: {estimated_capacity:.0f} transactions/{selected_granularity.lower()}")
print(f"   Current Utilization: {capacity_utilization:.1f}%")

if capacity_utilization > 80:
    print(f"   ⚠️  ALERT: Approaching capacity limits! Recommend infrastructure scaling.")
elif capacity_utilization > 60:
    print(f"   ⚠️  WARNING: Moderate capacity utilization. Monitor for future growth.")
else:
    print(f"   ✓ HEALTHY: Adequate capacity headroom available.")

# System Health Score
print(f"\n🏥 SYSTEM HEALTH SCORE:")

health_metrics = {
    "Completion Rate": metrics['current']['received_vs_eval_rate'] >= 95,
    "Evaluation Time": metrics['current']['avg_eval_time_sec'] <= 5.0,
    "Low Anomalies": len(anomalies) == 0,
    "Capacity": capacity_utilization <= 80
}

health_score = (sum(health_metrics.values()) / len(health_metrics)) * 100
health_status = "🟢 HEALTHY" if health_score >= 75 else "🟡 DEGRADED" if health_score >= 50 else "🔴 CRITICAL"

print(f"   Overall Score: {health_score:.0f}% {health_status}")
print(f"\n   Component Scores:")
for metric, status in health_metrics.items():
    status_icon = "✓" if status else "✗"
    print(f"     {status_icon} {metric}")

# Recommendations
print(f"\n💡 OPERATIONAL RECOMMENDATIONS:")

recommendations = []

if metrics['current']['received_vs_eval_rate'] < 95:
    recommendations.append("1. URGENT: Investigate evaluation processing delays")
    recommendations.append("   - Check worker node CPU/memory utilization")
    recommendations.append("   - Review rule evaluation engine performance")
    recommendations.append("   - Consider horizontal scaling of evaluation workers")

if metrics['current']['avg_eval_time_sec'] > 5:
    recommendations.append("1. Optimize transaction evaluation rules")
    recommendations.append("   - Profile slow-running rules")
    recommendations.append("   - Consider caching frequently accessed data")
    recommendations.append("   - Implement lazy evaluation where possible")

if capacity_utilization > 80:
    recommendations.append("1. Plan capacity expansion")
    recommendations.append("   - Forecast peak load growth trajectory")
    recommendations.append("   - Schedule infrastructure provisioning")
    recommendations.append("   - Consider CDN or load balancing improvements")

if len(anomalies) > 0:
    recommendations.append("1. Address detected anomalies immediately")
    for anomaly in anomalies:
        recommendations.append(f"   - {anomaly['type']}: {anomaly['description']}")

if not recommendations:
    recommendations.append("✓ System operating optimally. Continue monitoring.")

for rec in recommendations:
    print(f"   {rec}")

# Create System Health Dashboard with Indicator Gauges
fig_health = go.Figure()

# Completion Rate Gauge
fig_health.add_trace(
    go.Indicator(
        mode="gauge+number+delta",
        value=metrics['current']['received_vs_eval_rate'],
        domain={'x': [0, 0.45], 'y': [0.55, 1]},
        title={'text': "Completion Rate (%)"},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': "green" if metrics['current']['received_vs_eval_rate'] >= 95 else "red"},
               'steps': [{'range': [0, 95], 'color': "lightgray"}],
               'threshold': {'line': {'color': "red", 'width': 4},
                           'thickness': 0.75,
                           'value': 95}},
        delta={'reference': metrics['previous']['received_vs_eval_rate']}
    )
)

# Evaluation Time Gauge
fig_health.add_trace(
    go.Indicator(
        mode="gauge+number+delta",
        value=metrics['current']['avg_eval_time_sec'],
        domain={'x': [0.55, 1], 'y': [0.55, 1]},
        title={'text': "Avg Eval Time (s)"},
        gauge={'axis': {'range': [0, 10]},
               'bar': {'color': "green" if metrics['current']['avg_eval_time_sec'] <= 5 else "red"},
               'steps': [{'range': [0, 5], 'color': "lightgray"}],
               'threshold': {'line': {'color': "red", 'width': 4},
                           'thickness': 0.75,
                           'value': 5}},
        delta={'reference': metrics['previous']['avg_eval_time_sec']}
    )
)

# Capacity Utilization Gauge
fig_health.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=capacity_utilization,
        domain={'x': [0, 0.45], 'y': [0, 0.45]},
        title={'text': "Capacity Use (%)"},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': "green" if capacity_utilization <= 80 else "orange" if capacity_utilization <= 90 else "red"},
               'steps': [{'range': [0, 80], 'color': "lightgray"}],
               'threshold': {'line': {'color': "red", 'width': 4},
                           'thickness': 0.75,
                           'value': 90}}
    )
)

# Health Score Gauge
fig_health.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=health_score,
        domain={'x': [0.55, 1], 'y': [0, 0.45]},
        title={'text': "Health Score (%)"},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': "green" if health_score >= 75 else "orange" if health_score >= 50 else "red"},
               'steps': [{'range': [0, 50], 'color': "lightcoral"},
                        {'range': [50, 75], 'color': "lightyellow"},
                        {'range': [75, 100], 'color': "lightgreen"}],
               'threshold': {'line': {'color': "darkred", 'width': 4},
                           'thickness': 0.75,
                           'value': 75}}
    )
)

fig_health.update_layout(height=600, title_text="TMS System Health Dashboard")
fig_health.show()

print("\n✓ Capacity analysis complete!")


CAPACITY PLANNING & SYSTEM HEALTH

📊 THROUGHPUT ANALYSIS:
   Average Throughput: 500 transactions/hourly
   Peak Throughput: 593 transactions/hourly
   Estimated System Capacity: 988 transactions/hourly
   Current Utilization: 60.0%
   ✓ HEALTHY: Adequate capacity headroom available.

🏥 SYSTEM HEALTH SCORE:
   Overall Score: 50% 🟡 DEGRADED

   Component Scores:
     ✗ Completion Rate
     ✓ Evaluation Time
     ✗ Low Anomalies
     ✓ Capacity

💡 OPERATIONAL RECOMMENDATIONS:
   1. URGENT: Investigate evaluation processing delays
      - Check worker node CPU/memory utilization
      - Review rule evaluation engine performance
      - Consider horizontal scaling of evaluation workers
   1. Address detected anomalies immediately
      - SLA_BREACH: Evaluation completion rate below 95% SLA in 10 period(s)



✓ Capacity analysis complete!
